# 🚨 HMG 공급망 리스크 탐지 RAG 시스템

뉴스 기사가 입력되면 영향을 받는 협력사를 자동으로 탐지하고, 그 이유를 정리해 출력합니다.

### 데이터 흐름
```
[뉴스 입력]
    ↓ 임베딩 유사도 검색
[관련 협력사 Retrieval]  +  [관련 수입 품목 Retrieval]
    ↓ Prompt 조합
[Claude API → 리스크 분석 보고서 생성]
```

### 사용 데이터
- `hmg_partners_900.csv` – 협력사 목록 (회사명, 업종, 비즈니스 요약)
- `import_2025.csv` – 2025년 수입 데이터 (국가, HS코드, 품목, 단가)
- `news.csv` – 뉴스 기사 (제목, 내용)


## 0. 패키지 설치

In [ ]:
!pip install sentence-transformers faiss-cpu anthropic pandas numpy tqdm -q

## 1. 라이브러리 임포트 & 설정

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import anthropic

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# ✏️  설정값: 필요에 따라 수정하세요
# ─────────────────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")

# 한국어 지원 다국어 임베딩 모델
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

# 검색 시 반환할 최대 문서 수
TOP_K_PARTNERS = 10   # 협력사 후보
TOP_K_IMPORTS  = 8    # 수입 품목 후보
TOP_K_NEWS     = 5    # 유사 뉴스 후보

# 데이터 파일 경로 (같은 디렉토리 또는 경로 수정)
PATH_PARTNERS = "hmg_partners_900.csv"
PATH_IMPORTS  = "import_2025.csv"
PATH_NEWS     = "news.csv"

print("✅ 설정 완료")

## 2. 데이터 로드 & 전처리

In [ ]:
# ── 협력사 데이터 ──────────────────────────────
df_partners = pd.read_csv(PATH_PARTNERS)
df_partners.columns = df_partners.columns.str.strip()
df_partners = df_partners.fillna("")
print(f"협력사: {len(df_partners)}개")
print(df_partners.head(3).to_string())
print()

In [ ]:
# ── 수입 데이터 ──────────────────────────────
df_imports = pd.read_csv(PATH_IMPORTS)
df_imports = df_imports.fillna(0)
print(f"수입 레코드: {len(df_imports)}개")
print(df_imports.head(3).to_string())
print()

# 국가 + 품목 기준으로 집계 (단가 평균, 수입금액 합)
df_imp_agg = (
    df_imports
    .groupby(["country_name", "hs_code", "item_name"], as_index=False)
    .agg(
        avg_unit_price=("unit_price", "mean"),
        total_value=("imp_value", "sum"),
        months=("period", "count")
    )
)
df_imp_agg = df_imp_agg[df_imp_agg["total_value"] > 0].reset_index(drop=True)
print(f"집계 후 수입 레코드: {len(df_imp_agg)}개")

In [ ]:
# ── 뉴스 데이터 ──────────────────────────────
df_news = pd.read_csv(PATH_NEWS)
if "Unnamed: 0" in df_news.columns:
    df_news = df_news.drop(columns=["Unnamed: 0"])
df_news = df_news.fillna("")
# 중복 제거
df_news = df_news.drop_duplicates(subset=["title"]).reset_index(drop=True)
print(f"뉴스: {len(df_news)}개")
print(df_news[["title", "pub_date"]].head(5).to_string())

## 3. 문서(Document) 코퍼스 구성

각 데이터를 임베딩 가능한 텍스트 문서로 변환합니다.

In [ ]:
def build_partner_doc(row):
    """협력사 → 임베딩용 텍스트"""
    return (
        f"회사명: {row['회사명']}\n"
        f"업종: {row['업종']}\n"
        f"비즈니스 요약: {row['비즈니스 요약']}"
    )

def build_import_doc(row):
    """수입 레코드 → 임베딩용 텍스트"""
    return (
        f"수입국: {row['country_name']}\n"
        f"HS코드: {row['hs_code']}\n"
        f"품목: {row['item_name']}\n"
        f"평균단가: {row['avg_unit_price']:.2f} USD/kg\n"
        f"총수입액: {row['total_value']:.0f} 천달러"
    )

def build_news_doc(row):
    """뉴스 기사 → 임베딩용 텍스트 (제목 + 본문 앞 500자)"""
    content_preview = str(row.get("content", ""))[:500]
    return f"제목: {row['title']}\n내용: {content_preview}"

# 문서 리스트 생성
partner_docs  = [build_partner_doc(r) for _, r in df_partners.iterrows()]
import_docs   = [build_import_doc(r) for _, r in df_imp_agg.iterrows()]
news_docs     = [build_news_doc(r)   for _, r in df_news.iterrows()]

print(f"협력사 문서: {len(partner_docs)}개")
print(f"수입 문서:   {len(import_docs)}개")
print(f"뉴스 문서:   {len(news_docs)}개")
print()
print("[협력사 문서 예시]")
print(partner_docs[0])
print()
print("[수입 문서 예시]")
print(import_docs[0])

## 4. 임베딩 모델 로드 & FAISS 인덱스 구축

In [ ]:
print(f"임베딩 모델 로딩: {EMBEDDING_MODEL}")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("✅ 모델 로드 완료")

In [ ]:
def build_faiss_index(docs, model, batch_size=64):
    """문서 리스트 → FAISS L2 인덱스 반환"""
    print(f"  임베딩 생성 중 ({len(docs)}개 문서)...")
    embeddings = model.encode(
        docs,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True   # cosine similarity 사용 위해 정규화
    )
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner Product = cosine (정규화된 벡터)
    index.add(embeddings)
    print(f"  ✅ FAISS 인덱스 완료 (dim={dim}, 문서수={index.ntotal})")
    return index, embeddings

print("[1/3] 협력사 인덱스 구축...")
partner_index, partner_embeddings = build_faiss_index(partner_docs, embedder)

print("[2/3] 수입품목 인덱스 구축...")
import_index, import_embeddings = build_faiss_index(import_docs, embedder)

print("[3/3] 뉴스 인덱스 구축...")
news_index, news_embeddings = build_faiss_index(news_docs, embedder)

print("\n✅ 모든 FAISS 인덱스 구축 완료")

## 5. 검색(Retrieval) 함수 정의

In [ ]:
def retrieve(query: str, index, docs: list, metadata_df: pd.DataFrame, top_k: int = 5):
    """
    쿼리 → FAISS 검색 → (score, doc_text, metadata_row) 리스트 반환
    """
    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "score": float(score),
                "doc": docs[idx],
                "meta": metadata_df.iloc[idx].to_dict()
            })
    return results


def retrieve_by_news(news_title: str, news_content: str = ""):
    """
    뉴스 제목+내용 → 관련 협력사 & 수입품목 검색
    """
    query = f"{news_title} {news_content[:300]}".strip()
    
    partners = retrieve(query, partner_index, partner_docs, df_partners, TOP_K_PARTNERS)
    imports  = retrieve(query, import_index,  import_docs,  df_imp_agg,  TOP_K_IMPORTS)
    
    return partners, imports


print("✅ 검색 함수 정의 완료")

# 동작 확인
test_q = "이란 전쟁으로 인한 유가 상승"
test_partners, test_imports = retrieve_by_news(test_q)
print(f"\n[테스트] 쿼리: '{test_q}'")
print(f"  검색된 협력사: {len(test_partners)}개 (상위 유사도: {test_partners[0]['score']:.3f})")
print(f"  검색된 수입품목: {len(test_imports)}개 (상위 유사도: {test_imports[0]['score']:.3f})")

## 6. Claude API 기반 리스크 분석 함수

In [ ]:
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """
당신은 현대자동차그룹(HMG) 구매/공급망 전문 리스크 분석 AI입니다.
주어진 뉴스와 검색된 협력사·수입 품목 데이터를 바탕으로, 
어떤 협력사가 영향을 받는지 판단하고 그 이유를 체계적으로 정리합니다.

분석 시 다음 원칙을 따릅니다:
1. 뉴스에서 핵심 리스크 요인(지정학적 이슈, 원자재 가격, 공급 차질 등)을 식별합니다.
2. 검색된 협력사 중 실제로 영향을 받을 가능성이 높은 업체를 선별합니다.
3. 수입 품목 데이터와 연계하여 원자재/소재 공급 리스크를 분석합니다.
4. 리스크 수준을 [높음/중간/낮음]으로 명확히 표시합니다.
5. 권고 조치(선제적 단가 협상, 대체 공급선 검토 등)를 제안합니다.
6. 근거가 부족한 연결고리는 추측임을 명시합니다.
"""

def format_retrieved_context(
    partners: list,
    imports: list,
    partner_score_threshold: float = 0.3,
    import_score_threshold: float = 0.3
) -> str:
    """검색 결과를 프롬프트용 컨텍스트 문자열로 변환"""
    
    # 임계값 이상 협력사만 포함
    filtered_partners = [p for p in partners if p["score"] >= partner_score_threshold]
    filtered_imports  = [i for i in imports  if i["score"] >= import_score_threshold]
    
    ctx = "[검색된 협력사 후보]\n"
    if filtered_partners:
        for i, p in enumerate(filtered_partners, 1):
            ctx += f"\n{i}. (유사도: {p['score']:.3f})\n{p['doc']}\n"
    else:
        ctx += "(유사도 기준 이상의 협력사 없음)\n"
    
    ctx += "\n[검색된 수입 품목 현황]\n"
    if filtered_imports:
        for i, imp in enumerate(filtered_imports, 1):
            ctx += f"\n{i}. (유사도: {imp['score']:.3f})\n{imp['doc']}\n"
    else:
        ctx += "(유사도 기준 이상의 수입 품목 없음)\n"
    
    return ctx


def analyze_risk(
    news_title: str,
    news_content: str = "",
    news_date: str = "",
    verbose: bool = True
) -> str:
    """
    뉴스 입력 → 리스크 분석 보고서 반환
    
    Parameters
    ----------
    news_title   : 뉴스 제목
    news_content : 뉴스 본문 (없으면 제목만 사용)
    news_date    : 발행일 (참고용)
    verbose      : 중간 과정 출력 여부
    
    Returns
    -------
    str : 리스크 분석 보고서 (마크다운 형식)
    """
    if verbose:
        print(f"🔍 뉴스 분석 시작: {news_title[:60]}...")
    
    # ── Step 1: Retrieval ──────────────────────
    partners, imports = retrieve_by_news(news_title, news_content)
    context = format_retrieved_context(partners, imports)
    
    if verbose:
        print(f"   협력사 후보 {len(partners)}개, 수입품목 {len(imports)}개 검색 완료")
    
    # ── Step 2: Prompt 구성 ────────────────────
    content_preview = news_content[:800] if news_content else "(본문 없음)"
    
    user_prompt = f"""다음 뉴스를 분석하여 영향 받는 HMG 협력사와 리스크 원인을 보고서 형식으로 작성해주세요.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📰 뉴스 정보
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
발행일: {news_date if news_date else '미상'}
제목: {news_title}
본문 (요약):
{content_preview}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 RAG 검색 결과 (협력사 & 수입품목 데이터)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{context}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 작성 요청
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
아래 형식으로 보고서를 작성해주세요:

## 1. 뉴스 핵심 리스크 요인
(뉴스에서 공급망에 영향을 줄 수 있는 핵심 요소 3~5가지)

## 2. 영향 받는 협력사 분석
(각 협력사별로: 회사명, 리스크 수준[높음/중간/낮음], 영향 받는 이유)
단, 임베딩 유사도가 높더라도 실제 연관성이 낮다면 제외하세요.

## 3. 수입 원자재/품목 리스크
(어떤 수입품목/국가가 영향을 받는지, 단가 변동 예상)

## 4. 공급망 연결 시나리오
(뉴스 → 수입품목 영향 → 협력사 영향 → 최종 생산 차질로 이어지는 인과관계 서술)

## 5. 권고 조치
(단기/중기 대응 방안, 우선순위 포함)

## 6. 분석 한계 및 주의사항
(데이터 부족이나 추측에 의한 부분 명시)
"""
    
    # ── Step 3: Claude API 호출 ────────────────
    if verbose:
        print("   Claude API 호출 중...")
    
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=3000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}]
    )
    
    report = response.content[0].text
    
    if verbose:
        print("   ✅ 분석 완료\n")
    
    return report


print("✅ 분석 함수 정의 완료")

## 7. 단일 뉴스 분석 실행

In [ ]:
# ── 분석할 뉴스 선택 (news.csv에서 가져오기) ──────────
# 직접 선택하거나 아래처럼 특정 인덱스 사용
sample_idx = 0  # ← 0, 1, 2 ... 변경해서 다른 뉴스 분석 가능

sample_news = df_news.iloc[sample_idx]
print(f"선택된 뉴스:\n제목: {sample_news['title']}")
print(f"날짜: {sample_news.get('pub_date', '미상')}")
print(f"본문(앞 200자): {str(sample_news.get('content', ''))[:200]}...")

In [ ]:
# ── 리스크 분석 실행 ──────────────────────────
report = analyze_risk(
    news_title   = sample_news["title"],
    news_content = str(sample_news.get("content", "")),
    news_date    = str(sample_news.get("pub_date", "")),
    verbose      = True
)

print("=" * 70)
print("📊 공급망 리스크 분석 보고서")
print("=" * 70)
print(report)

## 8. 사용자 정의 뉴스 입력 분석

In [ ]:
# ── 직접 뉴스 텍스트를 입력해서 분석 ────────────
custom_title = "중동 분쟁 장기화로 알루미늄·철강 원자재 가격 30% 급등"
custom_content = """
미국-이란 갈등이 심화되면서 중동발 원자재 공급 불안이 가속화되고 있다.
호르무즈 해협을 통한 원유 및 알루미늄 원자재 수송이 차질을 빚고 있으며,
런던금속거래소(LME)에서 알루미늄 현물가가 전월 대비 28% 급등했다.
철강 슬래브와 캐스팅 얼로이 가격도 동반 상승세를 보이고 있어
자동차 부품 제조업체들의 원가 부담이 크게 증가할 것으로 예상된다.
특히 단조, 주조, 금속 가공 업종에 직격탄이 예상된다.
"""

custom_report = analyze_risk(
    news_title=custom_title,
    news_content=custom_content,
    verbose=True
)

print("=" * 70)
print("📊 공급망 리스크 분석 보고서 (사용자 정의 뉴스)")
print("=" * 70)
print(custom_report)

## 9. 배치 분석: 여러 뉴스 한꺼번에 처리

In [ ]:
def batch_analyze(
    news_df: pd.DataFrame,
    n_news: int = 5,
    filter_keyword: str = None,
    score_threshold: float = 0.35
) -> pd.DataFrame:
    """
    여러 뉴스를 배치로 분석하여 결과 DataFrame 반환
    
    Parameters
    ----------
    news_df         : 뉴스 DataFrame
    n_news          : 분석할 뉴스 수
    filter_keyword  : 제목 필터링 키워드 (None이면 전체)
    score_threshold : 협력사 포함 최소 유사도
    """
    if filter_keyword:
        subset = news_df[news_df["title"].str.contains(filter_keyword, na=False)].head(n_news)
    else:
        subset = news_df.head(n_news)
    
    results = []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="뉴스 분석"):
        # 빠른 검색 결과만 수집 (API 호출 없이)
        partners, imports = retrieve_by_news(
            row["title"], str(row.get("content", ""))
        )
        high_risk_partners = [
            p["meta"]["회사명"] for p in partners if p["score"] >= score_threshold
        ]
        top_countries = list({
            i["meta"]["country_name"] for i in imports if i["score"] >= score_threshold
        })
        top_items = list({
            i["meta"]["item_name"] for i in imports if i["score"] >= score_threshold
        })
        
        results.append({
            "뉴스 제목": row["title"],
            "발행일": row.get("pub_date", ""),
            "영향 협력사 수": len(high_risk_partners),
            "영향 협력사 목록": ", ".join(high_risk_partners[:5]),
            "관련 수입국": ", ".join(top_countries[:3]),
            "관련 품목": ", ".join(top_items[:3]),
            "최고 유사도(협력사)": partners[0]["score"] if partners else 0,
        })
    
    return pd.DataFrame(results)


# 키워드 필터링 없이 상위 5개 뉴스 배치 분석
batch_df = batch_analyze(df_news, n_news=5)
print("\n[배치 분석 결과]")
pd.set_option('display.max_colwidth', 40)
print(batch_df.to_string(index=False))

## 10. 특정 협력사 중심 역방향 분석

특정 협력사를 입력하면, 그 협력사에 영향을 줄 수 있는 뉴스를 찾습니다.

In [ ]:
def find_news_for_partner(company_name: str, top_k: int = 5) -> pd.DataFrame:
    """
    협력사명 → 관련 리스크 뉴스 탐색
    """
    # 협력사 정보 조회
    partner_row = df_partners[df_partners["회사명"].str.contains(company_name, na=False)]
    if partner_row.empty:
        print(f"'{company_name}' 협력사를 찾을 수 없습니다.")
        return pd.DataFrame()
    
    p = partner_row.iloc[0]
    query = f"{p['회사명']} {p['업종']} {p['비즈니스 요약']}"
    
    print(f"\n[협력사 정보]")
    print(f"  회사명: {p['회사명']}")
    print(f"  업종: {p['업종']}")
    print(f"  비즈니스: {p['비즈니스 요약']}")
    print()
    
    # 관련 뉴스 검색
    news_results = retrieve(query, news_index, news_docs, df_news, top_k)
    
    rows = []
    for r in news_results:
        rows.append({
            "유사도": round(r["score"], 3),
            "뉴스 제목": r["meta"]["title"],
            "발행일": r["meta"].get("pub_date", ""),
        })
    
    return pd.DataFrame(rows)


# 사용 예시: 협력사명 일부로 검색
result_df = find_news_for_partner("현대트랜시스")
if not result_df.empty:
    print("[관련 뉴스 탐색 결과]")
    pd.set_option('display.max_colwidth', 60)
    print(result_df.to_string(index=False))

In [ ]:
# 역방향 분석 후 전체 리스크 보고서 생성
company_to_analyze = "현대트랜시스"  # ← 여기서 협력사명 변경

news_for_partner = find_news_for_partner(company_to_analyze, top_k=3)

if not news_for_partner.empty:
    top_news_title = news_for_partner.iloc[0]["뉴스 제목"]
    top_news_row = df_news[df_news["title"] == top_news_title]
    
    if not top_news_row.empty:
        top_row = top_news_row.iloc[0]
        partner_report = analyze_risk(
            news_title=top_row["title"],
            news_content=str(top_row.get("content", "")),
            news_date=str(top_row.get("pub_date", "")),
            verbose=True
        )
        print("=" * 70)
        print(f"📊 '{company_to_analyze}' 관련 리스크 보고서")
        print("=" * 70)
        print(partner_report)

## 11. 대화형 인터페이스 (Interactive Mode)

셀을 실행하면 뉴스 제목을 직접 입력하여 실시간 분석할 수 있습니다.

In [ ]:
def interactive_rag():
    """
    대화형 RAG 실행 루프
    'quit' 입력 시 종료
    """
    print("" * 2)
    print("="*60)
    print("🚨 HMG 공급망 리스크 탐지 시스템")
    print("="*60)
    print("뉴스 제목을 입력하면 영향 받는 협력사를 분석합니다.")
    print("종료하려면 'quit' 입력\n")
    
    while True:
        title = input("📰 뉴스 제목: ").strip()
        if title.lower() in ["quit", "exit", "q"]:
            print("종료합니다.")
            break
        if not title:
            continue
        
        content = input("📄 뉴스 본문 (없으면 Enter): ").strip()
        
        report = analyze_risk(
            news_title=title,
            news_content=content,
            verbose=True
        )
        
        print("\n" + "="*60)
        print(report)
        print("="*60 + "\n")


# 대화형 모드 실행 (주석 해제 후 사용)
# interactive_rag()

## 12. 인덱스 저장 & 로드 (재실행 시 속도 향상)

In [ ]:
import pickle

def save_indices(save_dir="./rag_indices"):
    """FAISS 인덱스와 메타데이터를 파일로 저장"""
    os.makedirs(save_dir, exist_ok=True)
    
    faiss.write_index(partner_index, f"{save_dir}/partner.index")
    faiss.write_index(import_index,  f"{save_dir}/import.index")
    faiss.write_index(news_index,    f"{save_dir}/news.index")
    
    with open(f"{save_dir}/docs_meta.pkl", "wb") as f:
        pickle.dump({
            "partner_docs": partner_docs,
            "import_docs":  import_docs,
            "news_docs":    news_docs,
            "df_partners":  df_partners,
            "df_imp_agg":   df_imp_agg,
            "df_news":      df_news,
        }, f)
    
    print(f"✅ 인덱스 저장 완료: {save_dir}/")


def load_indices(save_dir="./rag_indices"):
    """저장된 인덱스 로드 (임베딩 재계산 불필요)"""
    global partner_index, import_index, news_index
    global partner_docs, import_docs, news_docs
    global df_partners, df_imp_agg, df_news
    
    partner_index = faiss.read_index(f"{save_dir}/partner.index")
    import_index  = faiss.read_index(f"{save_dir}/import.index")
    news_index    = faiss.read_index(f"{save_dir}/news.index")
    
    with open(f"{save_dir}/docs_meta.pkl", "rb") as f:
        data = pickle.load(f)
    
    partner_docs = data["partner_docs"]
    import_docs  = data["import_docs"]
    news_docs    = data["news_docs"]
    df_partners  = data["df_partners"]
    df_imp_agg   = data["df_imp_agg"]
    df_news      = data["df_news"]
    
    print(f"✅ 인덱스 로드 완료: {save_dir}/")


# 저장 실행
save_indices()

---
## 📌 사용 가이드

| 목적 | 사용 셀 |
|------|--------|
| 특정 뉴스 리스크 분석 | Section 7 `analyze_risk()` |
| 직접 뉴스 텍스트 입력 | Section 8 커스텀 입력 |
| 여러 뉴스 한번에 분석 | Section 9 `batch_analyze()` |
| 협력사 → 관련 뉴스 탐색 | Section 10 `find_news_for_partner()` |
| 실시간 대화형 입력 | Section 11 `interactive_rag()` |
| 인덱스 재사용 (속도 향상) | Section 12 `load_indices()` |

### 파라미터 튜닝
- `TOP_K_PARTNERS` / `TOP_K_IMPORTS`: 검색 후보 수 조절
- `partner_score_threshold`: 유사도 기준값 조절 (0.3~0.5 권장)
- `EMBEDDING_MODEL`: 다른 한국어 모델로 교체 가능 (`jhgan/ko-sroberta-multitask` 등)
